In [1]:
import pandas as pd
import numpy as np
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Load dataset
df = pd.read_csv("sample_credit_risk.csv")
df.head()

,income,age,loan_amount,loan_term,credit_score,employment_type,city_tier,default
0,698.75,38.0,1230.28,13.0,805.0,Salaried,1.0,1
1,356.72,53.0,517.90,24.0,742.0,Salaried,2.0,1
2,873.19,32.0,312.21,50.0,633.0,Self-Employed,2.0,1
3,960.27,30.0,1129.91,57.0,716.0,Salaried,3.0,0
4,226.20,34.0,972.30,40.0,698.0,Salaried,1.0,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   income           1471 non-null   float64
 1   age              1469 non-null   float64
 2   loan_amount      1473 non-null   float64
 3   loan_term        1465 non-null   float64
 4   credit_score     1477 non-null   float64
 5   employment_type  1469 non-null   object 
 6   city_tier        1470 non-null   float64
 7   default          1500 non-null   int64  
dtypes: float64(6), int64(1), object(1)
memory usage: 93.9+ KB


In [4]:
df.isnull().sum()

income             29
age                31
loan_amount        27
loan_term          35
credit_score       23
employment_type    31
city_tier          30
default             0
dtype: int64

#### Replacing Missing Values

In [5]:
# Numerical columns
num_cols = ['income', 'age', 'loan_amount', 'loan_term', 'credit_score']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [6]:
# Categorical column
str_cols = ['city_tier', 'employment_type']

for col in str_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
df.isnull().sum()

income             0
age                0
loan_amount        0
loan_term          0
credit_score       0
employment_type    0
city_tier          0
default            0
dtype: int64

In [8]:
# -----------------------------
# Data Preparation
# -----------------------------

In [9]:
# Features & target
X = df.drop("default", axis=1)
y = df["default"]

categorical_cols = ["employment_type", "city_tier"]
numerical_cols = ["income", "age", "loan_amount", "loan_term", "credit_score"]

In [10]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

In [11]:
# Model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500))
])

In [12]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Train model
model.fit(X_train, y_train)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
# Predictions
y_pred = model.predict(X_test)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

In [24]:
print("accuracy: ", round(accuracy,3),"\n")
print("report: ", report)
print("Confusion Matrix: ","\n", cm)

accuracy:  0.833 

report:                precision    recall  f1-score   support

           0       0.62      0.34      0.44        58
           1       0.86      0.95      0.90       242

    accuracy                           0.83       300
   macro avg       0.74      0.65      0.67       300
weighted avg       0.81      0.83      0.81       300

Confusion Matrix:  
 [[ 20  38]
 [ 12 230]]


In [25]:
# -----------------------------
# EDA Functions
# -----------------------------

def eda_summary():
    return df.describe().to_string()


def eda_missing():
    return df.isnull().sum().to_string()


def eda_distribution():
    return df.groupby("default").mean(numeric_only=True).to_string()

In [26]:
# -----------------------------
# Model Results
# -----------------------------

def model_results():
    result = f"Accuracy: {accuracy:.4f}\n\n"
    result += "Classification Report:\n" + report + "\n"
    result += "Confusion Matrix:\n" + str(cm)
    return result

In [27]:
# -----------------------------
# Prediction Function
# -----------------------------

def predict(income, age, loan_amount, loan_term, credit_score, employment_type, city_tier):
    input_df = pd.DataFrame({
        "income": [income],
        "age": [age],
        "loan_amount": [loan_amount],
        "loan_term": [loan_term],
        "credit_score": [credit_score],
        "employment_type": [employment_type],
        "city_tier": [city_tier]
    })
    
    pred = model.predict(input_df)[0]
    prob = model.predict_proba(input_df)[0][1]
    
    return f"Prediction (Default=1): {pred}\nProbability of Default: {prob:.2f}"

In [28]:
# -----------------------------
# Gradio Interface
# -----------------------------

with gr.Blocks(title="Credit Risk Analysis") as app:
    gr.Markdown("# Credit Risk Analysis Dashboard")

    with gr.Tab("EDA"):
        gr.Markdown("## Exploratory Data Analysis")
        btn1 = gr.Button("Summary Stats")
        out1 = gr.Textbox(label="Summary")
        btn1.click(eda_summary, outputs=out1)

        btn2 = gr.Button("Missing Values")
        out2 = gr.Textbox(label="Missing")
        btn2.click(eda_missing, outputs=out2)

        btn3 = gr.Button("Default Distribution")
        out3 = gr.Textbox(label="Distribution")
        btn3.click(eda_distribution, outputs=out3)

    with gr.Tab("Model Outcomes"):
        gr.Markdown("## Model Performance")
        btn4 = gr.Button("Show Results")
        out4 = gr.Textbox(label="Model Output")
        btn4.click(model_results, outputs=out4)

    with gr.Tab("Prediction"):
        gr.Markdown("## Predict Credit Default")

        income_in = gr.Number(label="Income")
        age_in = gr.Number(label="Age")
        loan_amount_in = gr.Number(label="Loan Amount")
        loan_term_in = gr.Number(label="Loan Term")
        credit_score_in = gr.Number(label="Credit Score")
        employment_type_in = gr.Dropdown(choices=list(df["employment_type"].unique()), label="Employment Type")
        city_tier_in = gr.Dropdown(choices=list(df["city_tier"].unique()), label="City Tier")

        btn5 = gr.Button("Predict")
        out5 = gr.Textbox(label="Prediction Result")

        btn5.click(
            predict,
            inputs=[income_in, age_in, loan_amount_in, loan_term_in, credit_score_in, employment_type_in, city_tier_in],
            outputs=out5
        )

# Launch app
app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
